# Quickstart: 4-bit Base Case Validation
**Runtime → Change runtime type → T4 GPU**

In [ ]:
# Cell 1: Setup
!git clone https://github.com/dhruvsyam123/recursive-transformers.git
%cd recursive-transformers
!pip install -q jax[cuda12] equinox optax jaxtyping numpy pyyaml matplotlib einops wandb

import jax
print(f"JAX devices: {jax.devices()}")
print(f"JAX backend: {jax.default_backend()}")

In [ ]:
# Cell 2: Test data pipeline
from src.data.karatsuba_trace import KaratsubaTraceGenerator
from src.data.dataset import MultiplicationDataset, DataConfig

# Quick sanity check
gen = KaratsubaTraceGenerator(base_case_bits=4)
trace = gen.generate(179, 214, 8)
print(f"179 * 214 = {trace.trace_product} (correct: {trace.verify()})")
print()

# Build 4-bit dataset
config = DataConfig(bit_widths=[4], algorithm='karatsuba', base_case_bits=4)
ds = MultiplicationDataset(config)
print(ds.summary())

# Peek at a batch
batch = next(ds.get_batch(split='train', batch_size=4))
print(f"\nBatch input_ids shape: {batch['input_ids'].shape}")
print(f"Batch target_ids shape: {batch['target_ids'].shape}")

In [ ]:
# Cell 3: Train 4-bit base case
from src.model.looped_transformer import create_model, compute_loss, count_parameters
import jax
import jax.numpy as jnp
import equinox as eqx
import optax

# Small model for 4-bit base case
key = jax.random.PRNGKey(42)
model = create_model(
    key=key,
    vocab_size=143,
    d_model=64,
    n_heads=2,
    d_ff=128,
    n_shared_layers=1,
    max_loop_iterations=1,
    max_seq_len=32,
    position_encoding_type='sinusoidal',
)

print(f"Model parameters: {count_parameters(model):,}")

optimizer = optax.adamw(learning_rate=3e-4, weight_decay=0.01)
opt_state = optimizer.init(eqx.filter(model, eqx.is_array))

@eqx.filter_jit
def train_step(model, opt_state, input_ids, target_ids, mask):
    def loss_fn(model):
        def forward(ids):
            positions = jnp.arange(ids.shape[0])
            return model(ids, positions, n_loops=1)
        logits = jax.vmap(forward)(input_ids)
        return compute_loss(logits, target_ids, mask)
    loss, grads = eqx.filter_value_and_grad(loss_fn)(model)
    updates, opt_state_new = optimizer.update(
        grads, opt_state, eqx.filter(model, eqx.is_array)
    )
    model_new = eqx.apply_updates(model, updates)
    return model_new, opt_state_new, loss

# Train
print("Training 4-bit base case...")
for epoch in range(50):
    epoch_loss = 0
    n_batches = 0
    for batch in ds.get_batch(split='train', batch_size=64):
        input_ids = jnp.array(batch['input_ids'])
        target_ids = jnp.array(batch['target_ids'])
        mask = jnp.array(batch['input_mask'])
        model, opt_state, loss = train_step(model, opt_state, input_ids, target_ids, mask)
        epoch_loss += float(loss)
        n_batches += 1
    if epoch % 5 == 0:
        print(f"Epoch {epoch:3d} | Loss: {epoch_loss/n_batches:.4f}")

print("Done!")

In [ ]:
# Cell 4: Evaluate 4-bit accuracy
correct = 0
total = 0
for batch in ds.get_batch(split='test', batch_size=64, shuffle=False):
    input_ids = jnp.array(batch['input_ids'])
    target_ids = jnp.array(batch['target_ids'])
    mask = jnp.array(batch['input_mask'])
    def forward(ids):
        positions = jnp.arange(ids.shape[0])
        return model(ids, positions, n_loops=1)
    logits = jax.vmap(forward)(input_ids)
    preds = jnp.argmax(logits, axis=-1)
    matches = (preds == target_ids) | (mask == 0)
    correct += int(jnp.all(matches, axis=-1).sum())
    total += len(batch['x'])

print(f"4-bit exact match: {correct}/{total} = {correct/total:.1%}")
print()
if correct/total == 1.0:
    print("PASS — base case perfect. Ready for 8-bit Karatsuba.")
else:
    print("FAIL — debug before proceeding. Paste output back to Claude.")